# Facturas GCP Pipeline - Testing Notebook
This notebook tests the complete XML → JSON → BigQuery pipeline with deduplication logic.

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install google-cloud-bigquery google-cloud-storage -q

## 2. Import Libraries

In [ ]:
import xml.etree.ElementTree as ET
import json
import hashlib
from datetime import datetime
from google.cloud import bigquery, storage
from google.oauth2 import service_account
import pandas as pd

## 3. Create Dummy XML Sample
Create test XML files with invoice data

In [ ]:
# Create dummy XML with 3 test invoices
dummy_xml = '''<?xml version="1.0" encoding="ISO-8859-1"?>
<SetDTE>
<DTE version="1.0">
  <Documento ID="Test1">
    <Encabezado>
      <IdDoc>
        <TipoDTE>33</TipoDTE>
        <Folio>1001</Folio>
        <FchEmis>2026-04-07</FchEmis>
      </IdDoc>
      <Emisor>
        <RUTEmisor>76741944-9</RUTEmisor>
        <RznSoc>TEST COMPANY 1</RznSoc>
      </Emisor>
      <Receptor>
        <RUTRecep>77384122-5</RUTRecep>
        <RznSocRecep>FK SPA</RznSocRecep>
      </Receptor>
      <Totales>
        <MntNeto>100000</MntNeto>
        <IVA>19000</IVA>
        <MntTotal>119000</MntTotal>
      </Totales>
    </Encabezado>
    <Detalle>
      <NroLinDet>1</NroLinDet>
      <NmbItem>Product 1</NmbItem>
      <QtyItem>10</QtyItem>
      <UnmdItem>UN</UnmdItem>
      <PrcItem>10000</PrcItem>
      <MontoItem>100000</MontoItem>
    </Detalle>
  </Documento>
<DTE version="1.0">
  <Documento ID="Test2">
    <Encabezado>
      <IdDoc>
        <TipoDTE>33</TipoDTE>
        <Folio>1002</Folio>
        <FchEmis>2026-04-07</FchEmis>
      </IdDoc>
      <Emisor>
        <RUTEmisor>77363855-1</RUTEmisor>
        <RznSoc>TEST COMPANY 2</RznSoc>
      </Emisor>
      <Receptor>
        <RUTRecep>77384122-5</RUTRecep>
        <RznSocRecep>FK SPA</RznSocRecep>
      </Receptor>
      <Totales>
        <MntNeto>50000</MntNeto>
        <IVA>9500</IVA>
        <MntTotal>59500</MntTotal>
      </Totales>
    </Encabezado>
    <Detalle>
      <NroLinDet>1</NroLinDet>
      <NmbItem>Product 2</NmbItem>
      <QtyItem>5</QtyItem>
      <UnmdItem>UN</UnmdItem>
      <PrcItem>10000</PrcItem>
      <MontoItem>50000</MontoItem>
    </Detalle>
  </Documento>
<DTE version="1.0">
  <Documento ID="Test3">
    <Encabezado>
      <IdDoc>
        <TipoDTE>33</TipoDTE>
        <Folio>1001</Folio>
        <FchEmis>2026-04-07</FchEmis>
      </IdDoc>
      <Emisor>
        <RUTEmisor>76741944-9</RUTEmisor>
        <RznSoc>TEST COMPANY 1</RznSoc>
      </Emisor>
      <Receptor>
        <RUTRecep>77384122-5</RUTRecep>
        <RznSocRecep>FK SPA</RznSocRecep>
      </Receptor>
      <Totales>
        <MntNeto>100000</MntNeto>
        <IVA>19000</IVA>
        <MntTotal>119000</MntTotal>
      </Totales>
    </Encabezado>
    <Detalle>
      <NroLinDet>1</NroLinDet>
      <NmbItem>Product 1</NmbItem>
      <QtyItem>10</QtyItem>
      <UnmdItem>UN</UnmdItem>
      <PrcItem>10000</PrcItem>
      <MontoItem>100000</MontoItem>
    </Detalle>
  </Documento>
</SetDTE>'''

# Save to file
with open('test_dummy.xml', 'w') as f:
    f.write(dummy_xml)

print("✅ Created dummy XML file: test_dummy.xml")
print(f"File size: {len(dummy_xml)} bytes")

## 4. XML Parser Function

In [ ]:
def element_to_dict(element):
    """Convert XML element to dictionary recursively"""
    result = {}
    for child in element:
        if len(child) == 0:
            result[child.tag] = child.text
        else:
            result[child.tag] = element_to_dict(child)
    return result

def parse_xml_to_rows(xml_file, filename):
    """Parse XML and extract document rows"""
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    rows = []
    for dte in root.findall('DTE'):
        documento = dte.find('Documento')
        if documento is not None:
            encabezado = documento.find('Encabezado')
            iddoc = encabezado.find('IdDoc')
            emisor = encabezado.find('Emisor')
            receptor = encabezado.find('Receptor')
            totales = encabezado.find('Totales')
            
            # Extract values
            tipo_dte = iddoc.findtext('TipoDTE')
            folio = iddoc.findtext('Folio')
            fecha_emis = iddoc.findtext('FchEmis')
            rut_emisor = emisor.findtext('RUTEmisor')
            rsn_emisor = emisor.findtext('RznSoc')
            rut_receptor = receptor.findtext('RUTRecep')
            rsn_receptor = receptor.findtext('RznSocRecep')
            mnt_neto = totales.findtext('MntNeto')
            iva = totales.findtext('IVA')
            mnt_total = totales.findtext('MntTotal')
            
            # Extract line items
            detalles = [element_to_dict(d) for d in documento.findall('Detalle')]
            
            # Create hash for deduplication
            hash_key = f"{folio}#{rut_emisor}"
            hash_documento = hashlib.md5(hash_key.encode()).hexdigest()
            
            row = {
                'file_timestamp': datetime.utcnow().isoformat(),
                'source_filename': filename,
                'tipo_dte': tipo_dte,
                'folio': folio,
                'fecha_emis': fecha_emis,
                'rut_emisor': rut_emisor,
                'rsn_emisor': rsn_emisor,
                'rut_receptor': rut_receptor,
                'rsn_receptor': rsn_receptor,
                'mnt_neto': int(mnt_neto) if mnt_neto else None,
                'iva': int(iva) if iva else None,
                'mnt_total': int(mnt_total) if mnt_total else None,
                'detalle_json': json.dumps(detalles),
                'documento_json': json.dumps(element_to_dict(documento)),
                'hash_documento': hash_documento,
                'is_duplicate': False,
            }
            rows.append(row)
    
    return rows

# Test parsing
rows = parse_xml_to_rows('test_dummy.xml', 'test_dummy.xml')
print(f"✅ Parsed {len(rows)} documents from XML")
print(f"\nFirst row (without JSON fields):\n")
first_row_display = {k: v for k, v in rows[0].items() if k not in ['detalle_json', 'documento_json']}
for key, value in first_row_display.items():
    print(f"  {key}: {value}")

## 5. Deduplication Logic Test

In [ ]:
# Check for duplicates in the test data
print("Deduplication Analysis:")
print("=" * 50)

hashes = {}
for i, row in enumerate(rows):
    hash_val = row['hash_documento']
    if hash_val not in hashes:
    hashes[hash_val] = []
    hashes[hash_val].append(i)

for hash_val, indices in hashes.items():
    if len(indices) > 1:
        print(f"\n⚠️  DUPLICATE FOUND (hash: {hash_val[:8]}...)")
        for idx in indices:
            print(f"   Document {idx}: Folio {rows[idx]['folio']} from {rows[idx]['rut_emisor']}")
    else:
        idx = indices[0]
        print(f"✅ Unique Document {idx}: Folio {rows[idx]['folio']} from {rows[idx]['rut_emisor']}")

## 6. BigQuery Connection Test
Test connection to BigQuery and check existing data

In [ ]:
# Initialize BigQuery client using Application Default Credentials
PROJECT_ID = "impasto-492602"
DATASET_ID = "logistica"
TABLE_ID = "facturas_raw"

try:
    client = bigquery.Client(project=PROJECT_ID)
    table = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}")
    print(f"✅ Connected to BigQuery")
    print(f"Dataset: {DATASET_ID}")
    print(f"Table: {TABLE_ID}")
    print(f"Schema: {len(table.schema)} fields")
except Exception as e:
    print(f"❌ Error connecting to BigQuery: {str(e)}")
    print("Make sure you have credentials configured (gcloud auth application-default login)")

## 7. Check Existing Data

In [ ]:
# Query existing data
query = f"""
SELECT 
    folio, 
    rut_emisor, 
    rsn_emisor,
    mnt_total,
    is_duplicate,
    processing_status,
    file_timestamp
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
LIMIT 10
"""

try:
    df = client.query(query).to_pandas()
    print(f"Existing records in BigQuery: {len(df)}")
    if len(df) > 0:
        print("\nFirst 5 records:")
        print(df.head())
    else:
        print("No records yet in the table")
except Exception as e:
    print(f"Error querying BigQuery: {str(e)}")

## 8. Insert Test Data into BigQuery
Insert the parsed rows and test deduplication on subsequent inserts

In [ ]:
# Insert rows into BigQuery
try:
    table = client.get_table(f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}")
    errors = client.insert_rows_json(table, rows)
    
    if errors:
        print(f"❌ Insert errors: {errors}")
    else:
        print(f"✅ Successfully inserted {len(rows)} rows into BigQuery")
except Exception as e:
    print(f"Error inserting rows: {str(e)}")

## 9. Query and Display Results

In [ ]:
# Query the results
query = f"""
SELECT 
    folio,
    rut_emisor,
    rsn_emisor,
    rut_receptor,
    mnt_total,
    is_duplicate,
    processing_status,
    source_filename
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
ORDER BY file_timestamp DESC
LIMIT 20
"""

try:
    df = client.query(query).to_pandas()
    print(f"Recent records:")
    print(df.to_string())
except Exception as e:
    print(f"Error querying: {str(e)}")

## 10. Deduplication Statistics

In [ ]:
# Get statistics
query = f"""
SELECT 
    processing_status,
    COUNT(*) as count,
    COUNT(DISTINCT hash_documento) as unique_hashes
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
GROUP BY processing_status
"""

try:
    stats_df = client.query(query).to_pandas()
    print("Processing Statistics:")
    print(stats_df.to_string())
    
    # Check for duplicates
    query_duplicates = f"""
    SELECT 
        folio,
        rut_emisor,
        rsn_emisor,
        COUNT(*) as occurrences,
        COUNT(CASE WHEN is_duplicate = True THEN 1 END) as duplicate_count
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    GROUP BY folio, rut_emisor, rsn_emisor
    HAVING COUNT(*) > 1
    """
    
    dup_df = client.query(query_duplicates).to_pandas()
    if len(dup_df) > 0:
        print("\n⚠️  Duplicate Documents Found:")
        print(dup_df.to_string())
    else:
        print("\n✅ No duplicates detected")
except Exception as e:
    print(f"Error: {str(e)}")